# TabDPT Classifier — DIMER end-to-end tutorial

**Profile:** `E2E`  
**Notebook specification:** DIMER Notebook Specification `1.0`  
**Repository code revision exercised:** `cef1f0ae4af14c7c27364a1f1a9d8f573af3504e`

**By the end of this notebook you will be able to:** verify the pinned model; load a public sample or gated bring your own data; condition the in-context classifier; evaluate against a majority-class baseline; run inference on separate new records; export machine-readable outputs and a DIMER artifact; and verify it across a fresh reconstruction boundary.

References: [README](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/README.md), [model card](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/MODEL_CARD.md), [dataset spec](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/TABULAR_CLASSIFICATION_DATASET_SPEC.md), [DIMER contract](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/DIMER_CONTRACT.md), [pinned upstream](https://github.com/layer6ai-labs/TabDPT-inference/tree/9cfb05e0a6bc380ae6c99c08adc8d50dacd4f246), [Layer6/TabDPT](https://huggingface.co/Layer6/TabDPT), [paper](https://arxiv.org/abs/2410.18164).

`fit()` fits preprocessing and registers labelled support context; it does **not** gradient-train TabDPT. This tutorial does not establish calibration, fairness, robustness, or production fitness. Public-sample metrics are sanity evidence only because pretraining overlap cannot be ruled out.

## Prerequisites and runtime contract

**Google Colab is the supported release path**, Python 3.11–3.13. Other Jupyter environments may work but are outside this conformance claim. GPU is recommended; CPU is supported. The demonstrated path uses `use_flash=False`, `compile_model=False`, no quantization, and framework/device-default precision. The immutable code SHA is retained by `anchors/notebook-spec-v1-code-20260910`. Uploaded data remains inside the Colab runtime and is not sent by this notebook to an external inference service; do not upload confidential, restricted, or sensitive data unless that Colab environment is authorized for it. The default support ceiling is 10,000 rows and this tutorial uses `context_size=512` within the supported 128–16,384 range. Seeds control splitting and stochastic ensemble/context selection, but device, CUDA/library builds, and kernels may still cause run-to-run numeric variation; bitwise cross-device identity is not promised.

In [ ]:
import sys
if "torch" in sys.modules: raise RuntimeError("Start from a fresh Google Colab runtime before installing dependencies.")
REPO_REVISION="cef1f0ae4af14c7c27364a1f1a9d8f573af3504e"; REPO_DIR="/content/tabdpt-classifier-pipeline"
!rm -rf "$REPO_DIR"
!git clone -q https://github.com/kurtvalcorza/tabdpt-classifier-pipeline.git "$REPO_DIR"
!git -C "$REPO_DIR" checkout -q "$REPO_REVISION"
!python -m pip install -q -r "$REPO_DIR/tutorials/requirements-colab.txt"
!python -m pip install -q --no-deps "$REPO_DIR"

## 1. Verify the runtime and model provenance

Report effective Python/framework/device identity and verify the immutable checkpoint SHA-256 before model construction. The Hugging Face model repository is used for data files only; model-repository remote Python code is not executed. Success proves identity/integrity, not quality.

In [ ]:
import importlib.metadata as md, platform, torch
from tabdpt_classifier_pipeline import *
from tabdpt_classifier_pipeline.dimer_runtime import DimerRuntimeConfig
print("Python",sys.version.split()[0],"Platform",platform.platform(),"torch",md.version("torch"),"tabdpt",md.version("tabdpt"))
print("Device",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU","CUDA",torch.version.cuda)
print("Assumptions: compile_model=False use_flash=False quantization=None precision=framework/device default")
weights=resolve_tabdpt_weights()
print(TABDPT_HF_REPO,TABDPT_HF_REVISION,TABDPT_WEIGHT_FILENAME,TABDPT_WEIGHT_SHA256,TABDPT_UPSTREAM_CODE_COMMIT,weights)

## 2. Load the default sample or bring your own data

Default data are scikit-learn's copy of the public [UCI Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast-cancer-wisconsin-diagnostic), documented by [`load_breast_cancer`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html). UCI lists **CC BY 4.0**. This is tutorial/sanity data, not benchmark evidence. `upload_single` expects one labelled CSV with a `target` classification column plus uniquely named feature columns; `upload_presplit` expects labelled `train.csv` and `val.csv` plus unlabelled `new.csv` with the same effective feature schema. Random splitting assumes rows are sufficiently independent for a row-wise split to be meaningful. For temporal, grouped, spatial, patient/device, panel, or other leakage-sensitive data, use preserved pre-split files instead.

In [ ]:
DATA_MODE="sample" # @param ["sample","upload_single","upload_presplit"]
TARGET_COLUMN="target"; DROP_COLUMNS=[]; SEED=42
import csv, shutil, pandas as pd
from pathlib import Path
from collections import Counter
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
D=Path("/content/tabdpt-data"); shutil.rmtree(D,ignore_errors=True); D.mkdir()
def read_csv(p):
    with p.open(encoding="utf-8-sig",newline="") as f: h=next(csv.reader(f),None)
    if not h: raise ValueError(f"{p.name} is empty")
    dup=[x for x,n in Counter(h).items() if n>1]
    if dup: raise ValueError(f"{p.name} has duplicate columns: {dup}")
    return pd.read_csv(p)
def require_labeled(df,name):
    if TARGET_COLUMN not in df.columns: raise ValueError(f"{name} is missing required target column {TARGET_COLUMN!r}")
    if df[TARGET_COLUMN].isna().any(): raise ValueError(f"{name} target contains missing values")
    if df.drop(columns=[TARGET_COLUMN,*DROP_COLUMNS],errors="ignore").shape[1]==0: raise ValueError(f"{name} has no effective feature columns")
if DATA_MODE=="sample":
    f=load_breast_cancer(as_frame=True).frame
    a,new_labeled=train_test_split(f,test_size=.1,random_state=SEED,stratify=f[TARGET_COLUMN])
    support,evaluation=train_test_split(a,test_size=2/9,random_state=SEED,stratify=a[TARGET_COLUMN])
    new_records=new_labeled.drop(columns=[TARGET_COLUMN]); DATA_PROVENANCE={"source":"UCI WDBC via scikit-learn","license":"CC BY 4.0"}
else:
    from google.colab import files
    u=files.upload()
    for n,b in u.items(): (D/Path(n).name).write_bytes(b)
    if DATA_MODE=="upload_single":
        ps=list(D.glob("*.csv"))
        if len(ps)!=1: raise ValueError("upload exactly one CSV")
        f=read_csv(ps[0]); require_labeled(f,ps[0].name)
        a,new_labeled=train_test_split(f,test_size=.1,random_state=SEED,stratify=f[TARGET_COLUMN])
        support,evaluation=train_test_split(a,test_size=2/9,random_state=SEED,stratify=a[TARGET_COLUMN])
        new_records=new_labeled.drop(columns=[TARGET_COLUMN])
    elif DATA_MODE=="upload_presplit":
        required_paths=[D/"train.csv",D/"val.csv",D/"new.csv"]
        missing_files=[p.name for p in required_paths if not p.is_file()]
        if missing_files: raise ValueError(f"upload_presplit is missing required files: {missing_files}")
        support,evaluation,new_records=map(read_csv,required_paths)
    else: raise ValueError("unsupported DATA_MODE")
    DATA_PROVENANCE={"source":"user-provided"}
for name,df in [("support",support),("evaluation",evaluation)]: require_labeled(df,name)
if TARGET_COLUMN in new_records.columns: raise ValueError("new data must be unlabelled")
sc=set(support[TARGET_COLUMN].astype(str)); ec=set(evaluation[TARGET_COLUMN].astype(str))
if len(sc)<2: raise ValueError("classification requires at least two support classes")
if ec-sc: raise ValueError(f"evaluation classes absent from support: {sorted(ec-sc)}")
if set(pd.util.hash_pandas_object(support,index=False)) & set(pd.util.hash_pandas_object(evaluation,index=False)): raise ValueError("support/evaluation overlap")
if len(support)>10000: raise ValueError("support exceeds DIMER default 10,000-row ceiling")
print(DATA_PROVENANCE,support.shape,evaluation.shape,new_records.shape)

## 3. Condition the model on support data

This is preprocessing fit + in-context support conditioning, not gradient training. Unknown categories use fitted unknown-category handling; no preprocessing is learned from evaluation or new input.

In [ ]:
TUTORIAL_RUNTIME=DimerRuntimeConfig(target_column=TARGET_COLUMN,drop_columns=tuple(DROP_COLUMNS),max_train_rows=10000,validation_split=.2,fine_tune=False,n_ensembles=2,context_size=512,batch_size=512,temperature=1.0,seed=SEED)
INFERENCE=TUTORIAL_RUNTIME.inference_kwargs()
pipe=TabDPTClassificationPipeline(model_weight_path=weights,compile_model=False,use_flash=False,seed=SEED)
pipe.fit(support,target_column=TARGET_COLUMN,drop_columns=DROP_COLUMNS,seed=SEED)
print("classes",pipe.class_labels_,"features",pipe.feature_encoder.feature_columns)
def check_inference_schema(df,name):
    effective=df.drop(columns=pipe.drop_columns_,errors="ignore"); required=list(pipe.feature_encoder.feature_columns)
    missing=[c for c in required if c not in effective.columns]; extra=[c for c in effective.columns if c not in required]
    if missing or extra: raise ValueError(f"{name} feature schema mismatch; missing={missing}, extra={extra}")
    for col,mapping in pipe.feature_encoder.category_maps.items():
        unseen=sorted({str(v) for v in effective[col].dropna().tolist()}-set(mapping))
        if unseen: print(f"WARNING: {name}.{col} has unseen categories handled by the fitted unknown-category code: {unseen[:10]}")
check_inference_schema(evaluation.drop(columns=[TARGET_COLUMN]),"evaluation")
check_inference_schema(new_records,"new_records")
if len(support)>INFERENCE["context_size"]: print(f"Context reduction active: support rows={len(support)}, context_size={INFERENCE['context_size']}; seeded balanced subsampling applies.")
else: print("Context reduction inactive for current support size.")

## 4. Evaluate and compare a majority-class baseline

Accuracy is discrete correctness; log loss scores true-class probability mass; binary ROC-AUC is ranking quality where defined. Scores are probability-normalized but not established as calibrated confidence. The default decision rule is argmax. This single holdout has no dispersion estimate.

In [ ]:
import json
from sklearn.metrics import accuracy_score
metrics=pipe.evaluate(evaluation,**INFERENCE)
majority=support[TARGET_COLUMN].astype(str).mode().iloc[0]
baseline=float(accuracy_score(evaluation[TARGET_COLUMN].astype(str),[majority]*len(evaluation)))
evaluation_report={"tutorialEvidenceOnly":True,"modelMetrics":metrics,"majorityClassBaseline":{"class":majority,"accuracy":baseline}}
print(json.dumps(evaluation_report,indent=2))

## 5. Run inference on separate new records

These records are separate from evaluation. Output preserves row identity, argmax prediction, and class-score columns in class order.

In [ ]:
scores=pipe.predict_proba(new_records,**INFERENCE); pred=pipe.predict(new_records,**INFERENCE)
prediction_table=pd.DataFrame({"row_id":new_records.index.astype(str),"prediction":pred.astype(str)})
for label in pipe.class_labels_: prediction_table[f"score_{label}"]=scores[label].to_numpy()
print(prediction_table.head())

## 6. Export machine-readable outputs, provenance, and artifact

The serving state includes labelled support context and fitted preprocessing, so the artifact inherits source-data confidentiality, licensing, retention, and disclosure obligations.

In [ ]:
import hashlib
from dataclasses import asdict
O=Path("/content/tabdpt-output"); A=O/"artifact"; A.mkdir(parents=True,exist_ok=True)
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
prediction_table.to_csv(O/"predictions.csv",index=False)
(O/"metrics.json").write_text(json.dumps(evaluation_report,indent=2))
context_path=A/"training_context.parquet"; support.to_parquet(context_path,index=False)
preprocessing_state=pipe.export_preprocessing_state()
runtime_config=asdict(TUTORIAL_RUNTIME)
runtime_config["drop_columns"] = list(runtime_config["drop_columns"])
manifest={"format":"tabdpt-dimer-context-v3","taskType":"tabular_classification","targetColumn":TARGET_COLUMN,"dropColumns":list(preprocessing_state["dropColumns"]),"classNames":list(pipe.class_labels_),"runtimeConfig":runtime_config,"preprocessing":preprocessing_state,"baseModel":{"repo":TABDPT_HF_REPO,"revision":TABDPT_HF_REVISION,"filename":TABDPT_WEIGHT_FILENAME,"sha256":TABDPT_WEIGHT_SHA256,"upstreamCodeCommit":TABDPT_UPSTREAM_CODE_COMMIT},"trainingContext":{"path":context_path.name,"sizeBytes":context_path.stat().st_size,"sha256":sha(context_path)}}
manifest_path=A/"artifact.json"; manifest_path.write_text(json.dumps(manifest,indent=2))
validate_dimer_artifact(manifest_path,strict_directory=True)
provenance={"repositoryRevision":REPO_REVISION,"profile":"E2E","data":DATA_PROVENANCE,"model":manifest["baseModel"],"runtimeConfig":runtime_config,"runtime":{"python":sys.version.split()[0],"torch":md.version("torch"),"tabdpt":md.version("tabdpt"),"device":torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU","cuda":torch.version.cuda,"use_flash":False,"compile_model":False,"quantization":None,"precision":"framework/device default"}}
(O/"provenance.json").write_text(json.dumps(provenance,indent=2))
print(O,manifest_path,context_path)

## 7. Verify the serialized artifact across a fresh reconstruction boundary

Discard producer objects/runtime controls, copy only serialized artifact files, validate them, reconstruct serving state, and derive inference controls exclusively from `validated_manifest['runtimeConfig']`. This exercises the downstream artifact boundary.

In [ ]:
import gc, shutil, numpy as np
probe=new_records.iloc[:min(10,len(new_records))].copy()
before_classes=pipe.predict(probe,**INFERENCE).astype(str).to_numpy(); before_scores=pipe.predict_proba(probe,**INFERENCE).to_numpy()
del pipe, INFERENCE, TUTORIAL_RUNTIME, SEED, runtime_config, weights
gc.collect()
R=Path("/content/tabdpt-reload"); shutil.rmtree(R,ignore_errors=True); R.mkdir()
shutil.copy2(manifest_path,R/"artifact.json"); shutil.copy2(context_path,R/"training_context.parquet")
validated_manifest,validated_context=validate_dimer_artifact(R/"artifact.json",strict_directory=True)
reload_runtime = validated_manifest["runtimeConfig"]
reload_inference = {k:reload_runtime[k] for k in ("n_ensembles","context_size","batch_size","temperature","seed")}
reloaded=TabDPTClassificationPipeline.load_artifact(R/"artifact.json",compile_model=False,use_flash=False,seed=reload_runtime["seed"])
after_classes=reloaded.predict(probe,**reload_inference).astype(str).to_numpy(); after_scores=reloaded.predict_proba(probe,**reload_inference).to_numpy()
assert np.array_equal(before_classes,after_classes)
assert np.allclose(before_scores,after_scores,rtol=1e-6,atol=1e-7)
print("fresh reconstruction boundary: PASS",validated_context)

## Interpretation, limits, and next steps

A successful run proves the pinned code/model path can validate data, condition the classifier, compute tutorial metrics, score separate records, export a production-shaped DIMER v3 artifact, and reproduce equivalent probe outputs after serialization. It does **not** prove domain accuracy, calibration, fairness, robustness, security, or production fitness. Clean Google Colab execution evidence for the exact release revision remains required; static CI is not execution evidence. For a next experiment, evaluate on a domain-valid independent labelled test set and assess calibration separately if downstream decisions depend on score magnitudes.